### Imports & Utility Functions

In [163]:
import yaml
import joblib
import numpy as np
import lightgbm as lgb
import optuna
import time
import os
import warnings
from sklearn.metrics import mean_squared_error, r2_score

# Silence warnings/logs (optional)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

# Optional: pyRAPL for energy measurement
try:
    import pyRAPL
    pyRAPL.setup()
    _has_rapl = True
except (ImportError, FileNotFoundError):
    _has_rapl = False

def load_config(path="config.yml"):
    with open(path, "r") as f:
        return yaml.safe_load(f)

print("Imports and utility functions loaded.")

Imports and utility functions loaded.


### Load Config & Data Splits

In [179]:
# Load YAML into a dictionary
cfg = load_config("config.yml")
print("Config keys:", cfg.keys())

# Load preprocessed train/val/test splits via joblib
X_tr, y_tr = joblib.load("data/processed/train.pkl")
X_val, y_val = joblib.load("data/processed/val.pkl")
X_te, y_te = joblib.load("data/processed/test.pkl")

print("Shapes →",
      "X_tr:", X_tr.shape, "y_tr:", y_tr.shape,
      "X_val:", X_val.shape, "y_val:", y_val.shape,
      "X_te:", X_te.shape, "y_te:", y_te.shape)

Config keys: dict_keys(['seed', 'optuna', 'lgbm', 'xgb', 'cat', 'mlp', 'cv', 'baseline', 'output'])
Shapes → X_tr: (151075, 153) y_tr: (151075,) X_val: (21583, 153) y_val: (21583,) X_te: (43165, 153) y_te: (43165,)


## Optuna-Tuned LightGBM

In [123]:
def run_optuna_lgb(cfg, X_tr, y_tr, X_val, y_val, X_te, y_te):
    """
    1) Tune LightGBM hyperparameters on (X_tr,y_tr) vs (X_val,y_val) via Optuna.
    2) Retrain best‐found model on (train+val), then evaluate on (X_te,y_te).
    3) Measure inference latency, model file size, and optional energy usage.
    Returns a dict of hyperparameters + val/test metrics + latency/size/energy.
    """

    # 1) Hyperparameter tuning
    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    def objective(trial):
        params = {
            "objective":        "regression",
            "metric":           "rmse",
            "learning_rate":    trial.suggest_float(
                                    "learning_rate",
                                    *cfg["lgbm"]["learning_rate"],
                                    log=True
                                ),
            "max_depth":        trial.suggest_int(
                                    "max_depth",
                                    *cfg["lgbm"]["max_depth"]
                                ),
            "subsample":        trial.suggest_float(
                                    "subsample",
                                    *cfg["lgbm"]["subsample"]
                                ),
            "colsample_bytree": trial.suggest_float(
                                    "colsample_bytree",
                                    *cfg["lgbm"]["colsample"]
                                ),
            "verbose": -1,
            "seed":    cfg["seed"]
        }
        bst = lgb.train(
            params,
            dtrain,
            num_boost_round=cfg["lgbm"]["n_estimators"],
            valid_sets=[dval],
            callbacks=[
                lgb.early_stopping(
                    stopping_rounds=cfg["lgbm"]["early_stopping_rounds"]
                ),
                lgb.log_evaluation(period=0)
            ],
        )
        preds_val = bst.predict(X_val, num_iteration=bst.best_iteration)
        return mean_squared_error(y_val, preds_val, squared=False)

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=cfg["seed"])
    )
    study.optimize(
        objective,
        n_trials=cfg["optuna"]["n_trials"],
        show_progress_bar=False
    )

    best_params = study.best_trial.params
    val_rmse    = study.best_value

    # 2) Retrain on (train + val)
    X_full = np.vstack([X_tr, X_val])
    y_full = np.concatenate([y_tr, y_val])
    d_full = lgb.Dataset(X_full, label=y_full)

    final_model = lgb.train(
        {**best_params, "objective": "regression", "metric": "rmse", "verbose": -1},
        d_full,
        num_boost_round=cfg["lgbm"]["n_estimators"]
    )

    # 3) Test metrics
    y_pred_test = final_model.predict(X_te)
    test_rmse = mean_squared_error(y_te, y_pred_test, squared=False)
    test_r2   = r2_score(y_te, y_pred_test)

    # 4) Inference latency (ms/sample)
    n_eval = min(2048, len(X_te))
    batch  = X_te[:n_eval]
    t0 = time.time()
    _  = final_model.predict(batch)
    latency_ms = (time.time() - t0) * 1000.0 / n_eval

    # 5) Energy usage (J via pyRAPL), if available
    energy_j = None
    if _has_rapl:
        meter = pyRAPL.Measurement("inference")
        meter.begin()
        _ = final_model.predict(batch)
        meter.end()
        energy_j = meter.result.pkg[0]

    # 6) Model size (MB)
    tmp_path = "tmp_lgbm_model.txt"
    final_model.save_model(tmp_path)
    size_mb = os.path.getsize(tmp_path) / (1024 ** 2)
    os.remove(tmp_path)

    return {
        "model":          "Optuna_LightGBM",
        "search":         "Optuna (TPE)",
        "tunable_params": ", ".join(best_params.keys()),
        **best_params,
        "val_rmse":       round(val_rmse, 3),
        "test_rmse":      round(test_rmse, 3),
        "test_r2":        round(test_r2, 4),
        "latency_ms":     round(latency_ms, 3),
        "model_size_mb":  round(size_mb, 3),
        "energy_j":       round(energy_j, 3) if energy_j is not None else None
    }

In [125]:
# Execute the tuning + retraining + measurements:
result_lgb = run_optuna_lgb(cfg, X_tr, y_tr, X_val, y_val, X_te, y_te)

# Pretty‐print the dictionary of results:
import pprint
pprint.pprint(result_lgb)

[I 2025-06-06 12:58:55,438] A new study created in memory with name: no-name-6af0019d-6aad-47a0-b424-0171dec86a4c


Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's rmse: 10464


[I 2025-06-06 12:58:58,212] Trial 0 finished with value: 10463.951261691565 and parameters: {'learning_rate': 0.005611516415334507, 'max_depth': 12, 'subsample': 0.8659969709057025, 'colsample_bytree': 0.7993292420985183}. Best is trial 0 with value: 10463.951261691565.


Training until validation scores don't improve for 50 rounds


[I 2025-06-06 12:58:59,764] Trial 1 finished with value: 14434.36283473835 and parameters: {'learning_rate': 0.0020513382630874496, 'max_depth': 4, 'subsample': 0.5290418060840998, 'colsample_bytree': 0.9330880728874675}. Best is trial 0 with value: 10463.951261691565.


...

Early stopping, best iteration is:
[411]	valid_0's rmse: 9531.66
Training until validation scores don't improve for 50 rounds


[I 2025-06-06 12:59:28,660] Trial 18 finished with value: 9617.152735401149 and parameters: {'learning_rate': 0.027600313749959474, 'max_depth': 6, 'subsample': 0.7485426968054263, 'colsample_bytree': 0.9397331285368563}. Best is trial 10 with value: 9319.666729255523.


Did not meet early stopping. Best iteration is:
[999]	valid_0's rmse: 9617.15
Training until validation scores don't improve for 50 rounds


[I 2025-06-06 12:59:30,297] Trial 19 finished with value: 9422.511051463267 and parameters: {'learning_rate': 0.05907333972036371, 'max_depth': 10, 'subsample': 0.956801292970232, 'colsample_bytree': 0.7764209924043369}. Best is trial 10 with value: 9319.666729255523.


Early stopping, best iteration is:
[658]	valid_0's rmse: 9422.51
{'colsample_bytree': 0.9775458067418015,
 'energy_j': None,
 'latency_ms': 0.01,
 'learning_rate': 0.0809056940601613,
 'max_depth': 8,
 'model': 'Optuna_LightGBM',
 'model_size_mb': 2.663,
 'search': 'Optuna (TPE)',
 'subsample': 0.5037736596581466,
 'test_r2': 0.9191,
 'test_rmse': 6742.297,
 'tunable_params': 'learning_rate, max_depth, subsample, colsample_bytree',
 'val_rmse': 9319.667}


## Optuna-Tuned XGBoost

In [127]:
def run_optuna_xgb(cfg, X_tr, y_tr, X_val, y_val, X_te, y_te):
    """
    1) Tune XGBoost hyperparameters on (X_tr,y_tr) vs (X_val,y_val) via Optuna.
    2) Retrain best XGBoost on (train+val), then evaluate on (X_te,y_te).
    3) Measure inference latency, model size, and optional energy.
    Returns a dict of hyperparameters + val/test metrics + latency/size/energy.
    """

    # 1) Hyperparameter Tuning via Optuna
    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dval   = xgb.DMatrix(X_val, label=y_val)

    def objective(trial):
        params = {
            "objective":        "reg:squarederror",
            "tree_method":      "hist",
            "learning_rate":    trial.suggest_float(
                                     "learning_rate",
                                     *cfg["xgb"]["learning_rate"],
                                     log=True
                                 ),
            "max_depth":        trial.suggest_int(
                                     "max_depth",
                                     *cfg["xgb"]["max_depth"]
                                 ),
            "subsample":        trial.suggest_float(
                                     "subsample",
                                     *cfg["xgb"]["subsample"]
                                 ),
            "colsample_bytree": trial.suggest_float(
                                     "colsample_bytree",
                                     *cfg["xgb"]["colsample"]
                                 ),
            "seed":             cfg["seed"]
        }

        # We still need to tell XGBoost when to early stop via the callback API:
        num_round = trial.suggest_int("n_estimators", *cfg["xgb"]["n_estimators"])
        bst = xgb.train(
            params,
            dtrain,
            num_boost_round=num_round,
            evals=[(dval, "valid")],
            callbacks=[
                xgb.callback.EarlyStopping(
                    rounds=cfg["xgb"]["early_stopping_rounds"], save_best=True
                ),
                xgb.callback.TrainingCallback()  # suppress extra output
            ],
            verbose_eval=False
        )

        preds_val = bst.predict(dval, iteration_range=(0, bst.best_iteration))
        return mean_squared_error(y_val, preds_val, squared=False)

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=cfg["seed"])
    )
    study.optimize(
        objective,
        n_trials=cfg["optuna"]["n_trials"],
        show_progress_bar=False
    )

    best_params = study.best_trial.params
    val_rmse    = study.best_value

    # 2) Final retraining on (train + val)
    X_full = np.vstack([X_tr, X_val])
    y_full = np.concatenate([y_tr, y_val])
    d_full = xgb.DMatrix(X_full, label=y_full)

    final_bst = xgb.train(
        best_params,
        d_full,
        num_boost_round=best_params["n_estimators"],
        verbose_eval=False
    )

    # 3) Test evaluation
    dtest    = xgb.DMatrix(X_te)
    y_pred_test = final_bst.predict(dtest)
    test_rmse = mean_squared_error(y_te, y_pred_test, squared=False)
    test_r2   = r2_score(y_te, y_pred_test)

    # 4) Inference latency (ms/sample)
    n_eval = min(2048, len(X_te))
    batch  = X_te[:n_eval]
    t0 = time.time()
    _  = final_bst.predict(xgb.DMatrix(batch))
    latency_ms = (time.time() - t0) * 1000.0 / n_eval

    # 5) Energy (optional)
    energy_j = None
    if _has_rapl:
        meter = pyRAPL.Measurement("inference")
        meter.begin()
        _ = final_bst.predict(xgb.DMatrix(batch))
        meter.end()
        energy_j = meter.result.pkg[0]

    # 6) Model size (MB)
    tmp_path = "tmp_xgb_model.json"
    final_bst.save_model(tmp_path)
    size_mb = os.path.getsize(tmp_path) / (1024 ** 2)
    os.remove(tmp_path)

    # 7) Return dictionary
    return {
        "model":            "Optuna_XGBoost",
        "search":           "Optuna (TPE)",
        "tunable_params":   ", ".join([k for k in best_params if k != "n_estimators"]),
        **{k: best_params[k] for k in best_params if k != "n_estimators"},
        "n_estimators":     best_params["n_estimators"],
        "val_rmse":         round(val_rmse, 3),
        "test_rmse":        round(test_rmse, 3),
        "test_r2":          round(test_r2, 4),
        "latency_ms":       round(latency_ms, 3),
        "model_size_mb":    round(size_mb, 3),
        "energy_j":         round(energy_j, 3) if energy_j is not None else None
    }

In [129]:
result_xgb = run_optuna_xgb(cfg, X_tr, y_tr, X_val, y_val, X_te, y_te)
import pprint; pprint.pprint(result_xgb)

[I 2025-06-06 13:02:54,982] A new study created in memory with name: no-name-4368edc0-be80-4a07-ab37-3063a240f59a
[I 2025-06-06 13:02:58,870] Trial 0 finished with value: 13349.14501139524 and parameters: {'learning_rate': 0.005611516415334507, 'max_depth': 12, 'subsample': 0.8659969709057025, 'colsample_bytree': 0.7993292420985183, 'n_estimators': 240}. Best is trial 0 with value: 13349.14501139524.
[I 2025-06-06 13:03:02,514] Trial 1 finished with value: 16100.038552132633 and parameters: {'learning_rate': 0.002051110418843397, 'max_depth': 3, 'subsample': 0.9330880728874675, 'colsample_bytree': 0.8005575058716043, 'n_estimators': 737}. Best is trial 0 with value: 13349.14501139524.
[I 2025-06-06 13:03:06,767] Trial 2 finished with value: 21682.195729601943 and parameters: {'learning_rate': 0.0010994335574766201, 'max_depth': 12, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.6061695553391381, 'n_estimators': 263}. Best is trial 0 with value: 13349.14501139524.
[I 2025-06-06 

{'colsample_bytree': 0.8421165132560784,
 'energy_j': None,
 'latency_ms': 0.015,
 'learning_rate': 0.041380401125610165,
 'max_depth': 6,
 'model': 'Optuna_XGBoost',
 'model_size_mb': 1.938,
 'n_estimators': 496,
 'search': 'Optuna (TPE)',
 'subsample': 0.5488360570031919,
 'test_r2': 0.9316,
 'test_rmse': 6201.893,
 'tunable_params': 'learning_rate, max_depth, subsample, colsample_bytree',
 'val_rmse': 8293.093}


## Optuna-Tuned CatBoost

In [131]:
def run_optuna_cat(cfg, X_tr, y_tr, X_val, y_val, X_te, y_te):
    """
    1) Tune CatBoost hyperparameters on (X_tr,y_tr) vs (X_val,y_val) via Optuna.
    2) Retrain best CatBoost on (train+val), then evaluate on (X_te,y_te).
    3) Measure inference latency, model size, and optional energy.
    Returns a dict of hyperparameters + val/test metrics + latency/size/energy.
    """

    # 1) Hyperparameter Tuning via Optuna
    train_pool = Pool(X_tr, label=y_tr)
    val_pool   = Pool(X_val, label=y_val)

    def objective(trial):
        params = {
            "iterations":       cfg["cat"]["iterations"],  # large upper bound
            "learning_rate":    trial.suggest_float(
                                     "learning_rate",
                                     *cfg["cat"]["learning_rate"]
                                 ),
            "depth":            trial.suggest_int(
                                     "depth",
                                     *cfg["cat"]["depth"]
                                 ),
            "subsample":        trial.suggest_float(
                                     "subsample",
                                     *cfg["cat"]["subsample"]
                                 ),
            "colsample_bylevel":trial.suggest_float(
                                     "colsample_bylevel",
                                     *cfg["cat"]["colsample"]
                                 ),
            "loss_function":    "RMSE",
            "verbose":          False,
            "random_seed":      cfg["seed"]
        }
        model = CatBoostRegressor(**params)
        model.fit(
            train_pool,
            eval_set=val_pool,
            early_stopping_rounds=cfg["cat"]["early_stopping_rounds"],
            verbose=False
        )
        preds_val = model.predict(X_val)
        return mean_squared_error(y_val, preds_val, squared=False)

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=cfg["seed"])
    )
    study.optimize(
        objective,
        n_trials=cfg["optuna"]["n_trials"],
        show_progress_bar=False
    )

    best_params = study.best_trial.params
    val_rmse    = study.best_value

    # 2) Final retraining on (train + val)
    X_full = np.vstack([X_tr, X_val])
    y_full = np.concatenate([y_tr, y_val])
    full_pool = Pool(X_full, label=y_full)

    final_model = CatBoostRegressor(
        iterations=cfg["cat"]["iterations"],
        learning_rate=best_params["learning_rate"],
        depth=best_params["depth"],
        subsample=best_params["subsample"],
        colsample_bylevel=best_params["colsample_bylevel"],
        loss_function="RMSE",
        random_seed=cfg["seed"],
        verbose=False
    )
    final_model.fit(full_pool)

    # 3) Test evaluation
    y_pred_test = final_model.predict(X_te)
    test_rmse = mean_squared_error(y_te, y_pred_test, squared=False)
    test_r2   = r2_score(y_te, y_pred_test)

    # 4) Inference latency (ms/sample)
    n_eval = min(2048, len(X_te))
    batch  = X_te[:n_eval]
    t0 = time.time()
    _  = final_model.predict(batch)
    latency_ms = (time.time() - t0) * 1000.0 / n_eval

    # 5) Energy usage (optional)
    energy_j = None
    if _has_rapl:
        meter = pyRAPL.Measurement("inference")
        meter.begin()
        _ = final_model.predict(batch)
        meter.end()
        energy_j = meter.result.pkg[0]

    # 6) Model size (MB)
    tmp_path = "tmp_cat_model.cbm"
    final_model.save_model(tmp_path)
    size_mb = os.path.getsize(tmp_path) / (1024 ** 2)
    os.remove(tmp_path)

    # 7) Return dictionary
    return {
        "model":            "Optuna_CatBoost",
        "search":           "Optuna (TPE)",
        "tunable_params":   "learning_rate, depth, subsample, colsample_bylevel",
        "learning_rate":    best_params["learning_rate"],
        "depth":            best_params["depth"],
        "subsample":        best_params["subsample"],
        "colsample_bylevel":best_params["colsample_bylevel"],
        "val_rmse":         round(val_rmse, 3),
        "test_rmse":        round(test_rmse, 3),
        "test_r2":          round(test_r2, 4),
        "latency_ms":       round(latency_ms, 3),
        "model_size_mb":    round(size_mb, 3),
        "energy_j":         round(energy_j, 3) if energy_j is not None else None
    }

In [133]:
result_cat = run_optuna_cat(cfg, X_tr, y_tr, X_val, y_val, X_te, y_te)
import pprint; pprint.pprint(result_cat)

[I 2025-06-06 13:07:39,318] A new study created in memory with name: no-name-93291bda-5e8e-4e42-964a-7eee7aeefe10
[I 2025-06-06 13:08:17,405] Trial 0 finished with value: 6339.75068033687 and parameters: {'learning_rate': 0.043708610696262626, 'depth': 10, 'subsample': 0.8659969709057025, 'colsample_bylevel': 0.7993292420985183}. Best is trial 0 with value: 6339.75068033687.
[I 2025-06-06 13:08:25,419] Trial 1 finished with value: 6931.026659787929 and parameters: {'learning_rate': 0.02404167763981929, 'depth': 5, 'subsample': 0.5290418060840998, 'colsample_bylevel': 0.9330880728874675}. Best is trial 0 with value: 6339.75068033687.
[I 2025-06-06 13:08:36,841] Trial 2 finished with value: 6368.435821801338 and parameters: {'learning_rate': 0.0641003510568888, 'depth': 8, 'subsample': 0.5102922471479012, 'colsample_bylevel': 0.9849549260809971}. Best is trial 0 with value: 6339.75068033687.
[I 2025-06-06 13:08:45,359] Trial 3 finished with value: 5794.821446886394 and parameters: {'lear

{'colsample_bylevel': 0.59286688856823,
 'depth': 6,
 'energy_j': None,
 'latency_ms': 0.03,
 'learning_rate': 0.08184445354356794,
 'model': 'Optuna_CatBoost',
 'model_size_mb': 2.124,
 'search': 'Optuna (TPE)',
 'subsample': 0.5911429287324143,
 'test_r2': 0.9459,
 'test_rmse': 5514.484,
 'tunable_params': 'learning_rate, depth, subsample, colsample_bylevel',
 'val_rmse': 5746.896}


## Optuna-Tuned MLP (Keras)

In [183]:
def run_optuna_mlp(cfg, X_tr, y_tr, X_val, y_val, X_te, y_te):
    """
    1) Tune a Keras MLP on (X_tr, y_tr) vs. (X_val, y_val) using Optuna.
       Hyperparameters: n_layers, units, learning_rate, dropout.
    2) Retrain the best MLP on (train + val), then evaluate on (X_te, y_te).
    3) Measure inference latency, model size, and (optionally) energy.
    Returns a dict containing:
      • model name
      • search method
      • tunable hyperparameters
      • chosen hyperparameter values
      • val_rmse, test_rmse, test_r2
      • latency_ms, model_size_mb, energy_j
    """

    # Helper: build a Keras MLP given an Optuna trial
    def build_model(trial):
        n_layers = trial.suggest_int("n_layers", *cfg["mlp"]["n_layers"])
        units    = trial.suggest_int("units",    *cfg["mlp"]["units"],    step=cfg["mlp"]["step_units"])
        lr       = trial.suggest_float(
                       "learning_rate",
                       *cfg["mlp"]["learning_rate"],
                       log=True
                   )
        dropout  = trial.suggest_float("dropout", *cfg["mlp"]["dropout"])

        model = keras.Sequential()
        for _ in range(n_layers):
            model.add(keras.layers.Dense(units, activation="relu"))
            model.add(keras.layers.Dropout(dropout))
        model.add(keras.layers.Dense(1, activation="linear"))

        model.compile(
            optimizer=keras.optimizers.Adam(lr),
            loss="mse",
            metrics=[keras.metrics.RootMeanSquaredError()]
        )
        return model

    # 1) Hyperparameter tuning
    def objective(trial):
        model = build_model(trial)
        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=cfg["mlp"]["max_epochs"],
            batch_size=cfg["mlp"]["batch_size"],
            callbacks=[keras.callbacks.EarlyStopping(
                patience=cfg["mlp"]["patience"],
                restore_best_weights=True
            )],
            verbose=0
        )
        val_loss, val_rmse = model.evaluate(X_val, y_val, verbose=0)
        return val_rmse

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=cfg["seed"])
    )
    study.optimize(
        objective,
        n_trials=cfg["optuna"]["n_trials"],
        show_progress_bar=False
    )

    best_params = study.best_trial.params
    val_rmse    = study.best_value

    # 2) Final retraining on (train + val)
    X_full = np.vstack([X_tr, X_val])
    y_full = np.concatenate([y_tr, y_val])

    final_model = build_model(study.best_trial)
    final_model.fit(
        X_full, y_full,
        epochs=cfg["mlp"]["max_epochs"],
        batch_size=cfg["mlp"]["batch_size"],
        callbacks=[keras.callbacks.EarlyStopping(
            patience=cfg["mlp"]["patience"],
            restore_best_weights=True
        )],
        verbose=0
    )

    # 3) Test evaluation
    y_pred_test = final_model.predict(X_te).ravel()
    test_rmse = mean_squared_error(y_te, y_pred_test, squared=False)
    test_r2   = r2_score(y_te, y_pred_test)

    # 4) Inference latency (ms/sample)
    n_eval = min(1024, len(X_te))
    batch  = X_te[:n_eval]
    t0 = time.time()
    _  = final_model.predict(batch)
    latency_ms = (time.time() - t0) * 1000.0 / n_eval

    # 5) Energy usage (optional)
    energy_j = None
    if _has_rapl:
        meter = pyRAPL.Measurement("inference")
        meter.begin()
        _ = final_model.predict(batch)
        meter.end()
        energy_j = meter.result.pkg[0]

    # 6) Model size (MB)
    tmp_path = "tmp_mlp_model.h5"
    final_model.save(tmp_path)
    size_mb = os.path.getsize(tmp_path) / (1024 ** 2)
    os.remove(tmp_path)

    # 7) Return a dictionary of everything
    return {
        "model":          "Optuna_MLP",
        "search":         "Optuna (TPE)",
        "tunable_params": "n_layers, units, learning_rate, dropout",
        "n_layers":       best_params["n_layers"],
        "units":          best_params["units"],
        "learning_rate":  best_params["learning_rate"],
        "dropout":        best_params["dropout"],
        "val_rmse":       round(val_rmse, 3),
        "test_rmse":      round(test_rmse, 3),
        "test_r2":        round(test_r2, 4),
        "latency_ms":     round(latency_ms, 3),
        "model_size_mb":  round(size_mb, 3),
        "energy_j":       round(energy_j, 3) if energy_j is not None else None
    }

In [185]:
result_mlp = run_optuna_mlp(cfg, X_tr, y_tr, X_val, y_val, X_te, y_te)
import pprint; pprint.pprint(result_mlp)

[I 2025-06-06 13:46:02,577] A new study created in memory with name: no-name-1041c030-3502-41db-bc47-72b8a79b7361
[I 2025-06-06 13:46:59,525] Trial 0 finished with value: 7548.509765625 and parameters: {'n_layers': 2, 'units': 256, 'learning_rate': 0.0029106359131330704, 'dropout': 0.2993292420985183}. Best is trial 0 with value: 7548.509765625.
[I 2025-06-06 13:47:16,711] Trial 1 finished with value: 24790.25390625 and parameters: {'n_layers': 1, 'units': 64, 'learning_rate': 0.00013066739238053285, 'dropout': 0.4330880728874676}. Best is trial 0 with value: 7548.509765625.
[I 2025-06-06 13:47:58,060] Trial 2 finished with value: 15377.1591796875 and parameters: {'n_layers': 2, 'units': 192, 'learning_rate': 0.00010994335574766199, 'dropout': 0.48495492608099716}. Best is trial 0 with value: 7548.509765625.
[I 2025-06-06 13:48:25,753] Trial 3 finished with value: 11331.64453125 and parameters: {'n_layers': 3, 'units': 64, 'learning_rate': 0.0002310201887845295, 'dropout': 0.0917022549

1349/1349 ━━━━━━━━━━━━━━━━━━━━ 0s 277us/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 432us/step


{'dropout': 0.24913407839085017,
 'energy_j': None,
 'latency_ms': 0.039,
 'learning_rate': 0.005268453116251217,
 'model': 'Optuna_MLP',
 'model_size_mb': 1.227,
 'n_layers': 3,
 'search': 'Optuna (TPE)',
 'test_r2': 0.8799,
 'test_rmse': 8216.566,
 'tunable_params': 'n_layers, units, learning_rate, dropout',
 'units': 192,
 'val_rmse': 7201.516}


In [192]:
import pandas as pd

# “result_lgb”, “result_xgb”, “result_cat”, “result_mlp”
# should already exist from your previous runs.

df = pd.DataFrame([
    result_lgb,
    result_xgb,
    result_cat,
    result_mlp
])

df

,model,search,tunable_params,learning_rate,max_depth,subsample,colsample_bytree,val_rmse,test_rmse,test_r2,latency_ms,model_size_mb,energy_j,n_estimators,depth,colsample_bylevel,n_layers,units,dropout
0,Optuna_LightGBM,Optuna (TPE),"learning_rate, max_depth, subsample, colsample...",0.080906,8.0,0.503774,0.977546,9319.667,6742.297,0.9191,0.010,2.663,None,NaN,NaN,NaN,NaN,NaN,NaN
1,Optuna_XGBoost,Optuna (TPE),"learning_rate, max_depth, subsample, colsample...",0.041380,6.0,0.548836,0.842117,8293.093,6201.893,0.9316,0.015,1.938,None,496.0,NaN,NaN,NaN,NaN,NaN
2,Optuna_CatBoost,Optuna (TPE),"learning_rate, depth, subsample, colsample_byl...",0.081844,NaN,0.591143,NaN,5746.896,5514.484,0.9459,0.030,2.124,None,NaN,6.0,0.592867,NaN,NaN,NaN
3,Optuna_MLP,Optuna (TPE),"n_layers, units, learning_rate, dropout",0.005268,NaN,NaN,NaN,7201.516,8216.566,0.8799,0.039,1.227,None,NaN,NaN,NaN,3.0,192.0,0.249134


In [15]:
import pandas as pd

# 1) manually assemble your four rows
rows = [
    {
        "model":            "Optuna_LightGBM",
        "search":           "Optuna (TPE)",
        "tunable_params":   "learning_rate, max_depth, subsample, colsample_bytree",
        "learning_rate":    0.080906,
        "max_depth":        8.0,
        "subsample":        0.503774,
        "colsample":        0.977546,
        "val_rmse":         9319.667,
        "test_rmse":        6742.297,
        "test_r2":          0.9191,
        "latency_ms":       0.010,
        "model_size_mb":    2.663,
        # hard-code LightGBM n_estimators → 1000
        "n_estimators":     1000,
        "n_layers":         "",
        "units":            "",
        "dropout":          ""
    },
    {
        "model":            "Optuna_XGBoost",
        "search":           "Optuna (TPE)",
        "tunable_params":   "learning_rate, max_depth, subsample, colsample_bytree",
        "learning_rate":    0.041380,
        "max_depth":        6.0,
        "subsample":        0.548836,
        "colsample":        0.842117,
        "val_rmse":         8293.093,
        "test_rmse":        6201.893,
        "test_r2":          0.9316,
        "latency_ms":       0.015,
        "model_size_mb":    1.938,
        # from your tuning csv
        "n_estimators":     496,
        "n_layers":         "",
        "units":            "",
        "dropout":          ""
    },
    {
        "model":            "Optuna_CatBoost",
        "search":           "Optuna (TPE)",
        "tunable_params":   "learning_rate, depth, subsample, colsample_bylevel",
        "learning_rate":    0.081844,
        "max_depth":        6.0,
        "subsample":        0.591143,
        "colsample":        0.592867,    # this was your colsample_bylevel
        "val_rmse":         5746.896,
        "test_rmse":        5514.484,
        "test_r2":          0.9459,
        "latency_ms":       0.030,
        "model_size_mb":    2.124,
        # hard-code CatBoost iterations→n_estimators → 2000
        "n_estimators":     2000,
        "n_layers":         "",
        "units":            "",
        "dropout":          ""
    },
    {
        "model":            "Optuna_MLP",
        "search":           "Optuna (TPE)",
        "tunable_params":   "n_layers, units, learning_rate, dropout",
        "learning_rate":    0.005268,
        "max_depth":        "",         # not applicable
        "subsample":        "",
        "colsample":        "",
        "val_rmse":         7201.516,
        "test_rmse":        8216.566,
        "test_r2":          0.8799,
        "latency_ms":       0.039,
        "model_size_mb":    1.227,
        "n_layers":         3.0,
        "units":            192.0,
        "dropout":          0.249134
    }
]

# 2) build DataFrame
df = pd.DataFrame(rows)

# 3) enforce column order
cols = [
    "model","search","tunable_params",
    "learning_rate","max_depth","subsample","colsample",
    "val_rmse","test_rmse","test_r2",
    "latency_ms","model_size_mb",
    "n_estimators","n_layers","units","dropout"
]
df = df[cols]

# 4) write to CSV
df.to_csv("phase2_comparison_results.csv", index=False)
print("Written results.csv:")
print(df.to_csv(index=False))

Written results.csv:
model,search,tunable_params,learning_rate,max_depth,subsample,colsample,val_rmse,test_rmse,test_r2,latency_ms,model_size_mb,n_estimators,n_layers,units,dropout
Optuna_LightGBM,Optuna (TPE),"learning_rate, max_depth, subsample, colsample_bytree",0.080906,8.0,0.503774,0.977546,9319.667,6742.297,0.9191,0.01,2.663,1000.0,,,
Optuna_XGBoost,Optuna (TPE),"learning_rate, max_depth, subsample, colsample_bytree",0.04138,6.0,0.548836,0.842117,8293.093,6201.893,0.9316,0.015,1.938,496.0,,,
Optuna_CatBoost,Optuna (TPE),"learning_rate, depth, subsample, colsample_bylevel",0.081844,6.0,0.591143,0.592867,5746.896,5514.484,0.9459,0.03,2.124,2000.0,,,
Optuna_MLP,Optuna (TPE),"n_layers, units, learning_rate, dropout",0.005268,,,,7201.516,8216.566,0.8799,0.039,1.227,,3.0,192.0,0.249134

